In [0]:
from pyspark.sql import functions as F

CATALOG = "rio_dataengineering"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")
spark.sql(f"USE SCHEMA {GOLD_SCHEMA}")

print("Gold transformation configuration ready.")

In [0]:
def write_gold_table(df, table_name: str) -> int:
    """
    Overwrite a Gold Delta table and return its row count.
    """
    target_table = f"{CATALOG}.{GOLD_SCHEMA}.{table_name}"

    row_count = df.count()

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

    return row_count


employees = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.employee_profile"
)

payroll = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.payroll"
)

training = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.training"
)

engagement = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.engagement"
)

leave = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.leave"
)

print("Silver source tables loaded.")

In [0]:
employee_kpis = (
    employees.agg(
        F.countDistinct("employee_id").alias("total_employees"),

        F.sum(
            F.when(F.col("is_active"), 1).otherwise(0)
        ).alias("active_employees"),

        F.sum(
            F.when(
                F.col("employment_status") == "Terminated",
                1
            ).otherwise(0)
        ).alias("terminated_employees"),

        F.round(
            F.avg(
                F.when(
                    F.col("is_active"),
                    F.col("age_years")
                )
            ),
            2
        ).alias("average_active_employee_age"),

        F.round(
            F.avg(
                F.when(
                    F.col("is_active"),
                    F.col("tenure_years")
                )
            ),
            2
        ).alias("average_active_tenure_years"),

        F.countDistinct("department_id")
        .alias("department_count"),

        F.countDistinct("location_id")
        .alias("location_count")
    )
)

payroll_kpis = (
    payroll.agg(
        F.round(F.sum("gross_pay"), 2)
        .alias("total_gross_pay"),

        F.round(F.sum("net_pay"), 2)
        .alias("total_net_pay"),

        F.round(F.avg("base_salary"), 2)
        .alias("average_monthly_base_salary")
    )
)

training_kpis = (
    training.agg(
        F.sum("training_hours")
        .alias("total_training_hours"),

        F.round(F.avg("score"), 2)
        .alias("average_training_score"),

        F.round(
            F.avg(F.col("passed").cast("integer")) * 100,
            2
        ).alias("training_pass_rate_pct")
    )
)

engagement_kpis = (
    engagement.agg(
        F.round(F.avg("overall_score"), 2)
        .alias("average_engagement_score"),

        F.countDistinct("employee_id")
        .alias("engagement_responses")
    )
)

leave_kpis = (
    leave.agg(
        F.sum(
            F.when(
                F.col("approved"),
                F.col("leave_days")
            ).otherwise(0)
        ).alias("approved_leave_days"),

        F.sum(
            F.when(
                ~F.col("approved"),
                1
            ).otherwise(0)
        ).alias("unapproved_leave_requests")
    )
)

workforce_overview = (
    employee_kpis
    .crossJoin(payroll_kpis)
    .crossJoin(training_kpis)
    .crossJoin(engagement_kpis)
    .crossJoin(leave_kpis)
    .withColumn("snapshot_date", F.current_date())
)

workforce_rows = write_gold_table(
    workforce_overview,
    "workforce_overview"
)

print(
    f"Loaded gold.workforce_overview: "
    f"{workforce_rows} row"
)

In [0]:
payroll_by_employee = (
    payroll
    .groupBy("employee_id")
    .agg(
        F.round(F.avg("base_salary"), 2)
        .alias("average_base_salary"),

        F.round(F.sum("gross_pay"), 2)
        .alias("total_gross_pay"),

        F.round(F.sum("net_pay"), 2)
        .alias("total_net_pay")
    )
)

training_by_employee = (
    training
    .groupBy("employee_id")
    .agg(
        F.sum("training_hours")
        .alias("total_training_hours"),

        F.round(F.avg("score"), 2)
        .alias("average_training_score"),

        F.sum(F.col("passed").cast("integer"))
        .alias("courses_passed"),

        F.count("*")
        .alias("courses_completed")
    )
)

engagement_by_employee = (
    engagement
    .groupBy("employee_id")
    .agg(
        F.round(F.avg("overall_score"), 2)
        .alias("average_engagement_score")
    )
)

leave_by_employee = (
    leave
    .groupBy("employee_id")
    .agg(
        F.sum(
            F.when(
                F.col("approved"),
                F.col("leave_days")
            ).otherwise(0)
        ).alias("approved_leave_days")
    )
)

print("Employee-level Gold aggregations prepared.")

In [0]:
employee_analytics = (
    employees
    .join(
        payroll_by_employee,
        "employee_id",
        "left"
    )
    .join(
        training_by_employee,
        "employee_id",
        "left"
    )
    .join(
        engagement_by_employee,
        "employee_id",
        "left"
    )
    .join(
        leave_by_employee,
        "employee_id",
        "left"
    )
    .fillna(
        {
            "total_training_hours": 0,
            "courses_passed": 0,
            "courses_completed": 0,
            "approved_leave_days": 0
        }
    )
)

employee_analytics_rows = write_gold_table(
    employee_analytics,
    "employee_analytics"
)

print(
    f"Loaded gold.employee_analytics: "
    f"{employee_analytics_rows} rows"
)

In [0]:
department_kpis = (
    employee_analytics
    .groupBy(
        "department_id",
        "department_name"
    )
    .agg(
        F.countDistinct("employee_id")
        .alias("total_employees"),

        F.sum(
            F.when(F.col("is_active"), 1).otherwise(0)
        ).alias("active_employees"),

        F.sum(
            F.when(
                F.col("employment_status") == "Terminated",
                1
            ).otherwise(0)
        ).alias("terminated_employees"),

        F.round(F.avg("age_years"), 2)
        .alias("average_age_years"),

        F.round(F.avg("tenure_years"), 2)
        .alias("average_tenure_years"),

        F.round(F.avg("average_base_salary"), 2)
        .alias("average_base_salary"),

        F.round(F.avg("average_engagement_score"), 2)
        .alias("average_engagement_score"),

        F.sum("total_training_hours")
        .alias("total_training_hours"),

        F.sum("approved_leave_days")
        .alias("approved_leave_days")
    )
    .withColumn(
        "termination_rate_pct",
        F.round(
            F.col("terminated_employees")
            / F.col("total_employees") * 100,
            2
        )
    )
    .withColumn("snapshot_date", F.current_date())
)

department_rows = write_gold_table(
    department_kpis,
    "department_kpis"
)

print(
    f"Loaded gold.department_kpis: "
    f"{department_rows} rows"
)

In [0]:
location_kpis = (
    employee_analytics
    .groupBy(
        "location_id",
        "city",
        "country",
        "region"
    )
    .agg(
        F.countDistinct("employee_id")
        .alias("total_employees"),

        F.sum(
            F.when(F.col("is_active"), 1).otherwise(0)
        ).alias("active_employees"),

        F.round(F.avg("tenure_years"), 2)
        .alias("average_tenure_years"),

        F.round(F.avg("average_base_salary"), 2)
        .alias("average_base_salary"),

        F.round(F.avg("average_engagement_score"), 2)
        .alias("average_engagement_score"),

        F.sum("total_training_hours")
        .alias("total_training_hours")
    )
    .withColumn("snapshot_date", F.current_date())
)

location_rows = write_gold_table(
    location_kpis,
    "location_kpis"
)

print(
    f"Loaded gold.location_kpis: "
    f"{location_rows} rows"
)

In [0]:
monthly_payroll_summary = (
    payroll
    .groupBy("payroll_month")
    .agg(
        F.countDistinct("employee_id")
        .alias("paid_employee_count"),

        F.round(F.sum("base_salary"), 2)
        .alias("total_base_salary"),

        F.round(F.sum("overtime"), 2)
        .alias("total_overtime"),

        F.round(F.sum("bonus"), 2)
        .alias("total_bonus"),

        F.round(F.sum("tax"), 2)
        .alias("total_tax"),

        F.round(F.sum("gross_pay"), 2)
        .alias("total_gross_pay"),

        F.round(F.sum("net_pay"), 2)
        .alias("total_net_pay"),

        F.round(F.avg("gross_pay"), 2)
        .alias("average_gross_pay")
    )
)

payroll_summary_rows = write_gold_table(
    monthly_payroll_summary,
    "monthly_payroll_summary"
)

print(
    f"Loaded gold.monthly_payroll_summary: "
    f"{payroll_summary_rows} rows"
)

In [0]:
training_summary = (
    training
    .groupBy("course_name")
    .agg(
        F.count("*")
        .alias("employees_trained"),

        F.sum("training_hours")
        .alias("total_training_hours"),

        F.round(F.avg("score"), 2)
        .alias("average_score"),

        F.round(
            F.avg(F.col("passed").cast("integer")) * 100,
            2
        ).alias("pass_rate_pct")
    )
)

training_summary_rows = write_gold_table(
    training_summary,
    "training_summary"
)


engagement_summary = (
    engagement
    .groupBy("survey_date")
    .agg(
        F.countDistinct("employee_id")
        .alias("responses"),

        F.round(F.avg("engagement_score"), 2)
        .alias("average_engagement_score"),

        F.round(F.avg("wellbeing_score"), 2)
        .alias("average_wellbeing_score"),

        F.round(F.avg("manager_score"), 2)
        .alias("average_manager_score"),

        F.round(F.avg("overall_score"), 2)
        .alias("average_overall_score"),

        F.sum(
            F.when(
                F.col("engagement_category") == "High",
                1
            ).otherwise(0)
        ).alias("high_engagement_responses"),

        F.sum(
            F.when(
                F.col("engagement_category") == "Medium",
                1
            ).otherwise(0)
        ).alias("medium_engagement_responses"),

        F.sum(
            F.when(
                F.col("engagement_category") == "Low",
                1
            ).otherwise(0)
        ).alias("low_engagement_responses")
    )
    .withColumn(
        "high_engagement_rate_pct",
        F.round(
            F.col("high_engagement_responses")
            / F.col("responses") * 100,
            2
        )
    )
)

engagement_summary_rows = write_gold_table(
    engagement_summary,
    "engagement_summary"
)


leave_summary = (
    leave
    .groupBy("leave_type")
    .agg(
        F.count("*")
        .alias("requests"),

        F.sum("leave_days")
        .alias("total_leave_days"),

        F.sum(
            F.when(F.col("approved"), 1).otherwise(0)
        ).alias("approved_requests"),

        F.round(
            F.avg(F.col("approved").cast("integer")) * 100,
            2
        ).alias("approval_rate_pct")
    )
)

leave_summary_rows = write_gold_table(
    leave_summary,
    "leave_summary"
)

print(
    f"Loaded gold.training_summary: "
    f"{training_summary_rows} rows"
)

print(
    f"Loaded gold.engagement_summary: "
    f"{engagement_summary_rows} rows"
)

print(
    f"Loaded gold.leave_summary: "
    f"{leave_summary_rows} rows"
)

In [0]:
EXPECTED_GOLD_ROWS = {
    "workforce_overview": 1,
    "department_kpis": 7,
    "monthly_payroll_summary": 12,
    "location_kpis": 8,
    "employee_analytics": 100,
    "training_summary": 5,
    "engagement_summary": 1,
    "leave_summary": 4
}

validation_results = []

for table_name, expected_rows in EXPECTED_GOLD_ROWS.items():
    df = spark.table(
        f"{CATALOG}.{GOLD_SCHEMA}.{table_name}"
    )

    actual_rows = df.count()

    validation_results.append(
        {
            "table_name": table_name,
            "expected_rows": expected_rows,
            "actual_rows": actual_rows,
            "column_count": len(df.columns),
            "status": (
                "PASS"
                if actual_rows == expected_rows
                else "FAIL"
            )
        }
    )

validation_df = spark.createDataFrame(validation_results)

display(validation_df.orderBy("table_name"))

failed_tables = [
    result["table_name"]
    for result in validation_results
    if result["status"] != "PASS"
]

if failed_tables:
    raise ValueError(
        f"Gold validation failed: {failed_tables}"
    )

print("Gold tables validated successfully.")

In [0]:
quality_checks = [
    (
        "workforce_overview_single_row",
        spark.table(
            f"{CATALOG}.{GOLD_SCHEMA}.workforce_overview"
        ).count() == 1
    ),
    (
        "employee_analytics_unique_employee_id",
        spark.table(
            f"{CATALOG}.{GOLD_SCHEMA}.employee_analytics"
        ).groupBy("employee_id")
         .count()
         .filter(F.col("count") > 1)
         .count() == 0
    ),
    (
        "department_kpis_no_missing_department",
        spark.table(
            f"{CATALOG}.{GOLD_SCHEMA}.department_kpis"
        ).filter(F.col("department_id").isNull())
         .count() == 0
    ),
    (
        "location_kpis_no_missing_location",
        spark.table(
            f"{CATALOG}.{GOLD_SCHEMA}.location_kpis"
        ).filter(F.col("location_id").isNull())
         .count() == 0
    ),
    (
        "payroll_totals_non_negative",
        spark.table(
            f"{CATALOG}.{GOLD_SCHEMA}.monthly_payroll_summary"
        ).filter(F.col("total_gross_pay") < 0)
         .count() == 0
    ),
    (
        "training_pass_rate_valid",
        spark.table(
            f"{CATALOG}.{GOLD_SCHEMA}.training_summary"
        ).filter(
            ~F.col("pass_rate_pct").between(0, 100)
        ).count() == 0
    ),
    (
        "leave_approval_rate_valid",
        spark.table(
            f"{CATALOG}.{GOLD_SCHEMA}.leave_summary"
        ).filter(
            ~F.col("approval_rate_pct").between(0, 100)
        ).count() == 0
    )
]

quality_results = [
    {
        "quality_check": check_name,
        "status": "PASS" if passed else "FAIL"
    }
    for check_name, passed in quality_checks
]

quality_df = spark.createDataFrame(quality_results)

display(quality_df.orderBy("quality_check"))

failed_checks = [
    check_name
    for check_name, passed in quality_checks
    if not passed
]

if failed_checks:
    raise ValueError(
        f"Gold quality checks failed: {failed_checks}"
    )

print("Gold transformations and quality checks completed successfully.")